In [1]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query, register_integrated_taxi_trips

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('storage_metrics')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 19:34:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [13]:
spark.sql("""select  started_at,finished_at,duration_seconds,dataset from delta.`s3a://delta-data/metadata/pipeline_executions` where stage = 'analytics_refresh' order by dataset""").show()

+--------------------+--------------------+----------------+--------------------+
|          started_at|         finished_at|duration_seconds|             dataset|
+--------------------+--------------------+----------------+--------------------+
|2026-09-27 19:10:...|2026-09-27 19:10:...|          10.938|air_quality_vs_de...|
|2026-09-27 19:28:...|2026-09-27 19:28:...|           6.775|air_quality_vs_de...|
|2026-09-27 19:09:...|2026-09-27 19:10:...|          11.398|average_trip_dist...|
|2026-09-27 19:28:...|2026-09-27 19:28:...|           26.16|average_trip_dist...|
|2026-09-27 19:09:...|2026-09-27 19:09:...|          13.226| monthly_zone_demand|
|2026-09-27 19:27:...|2026-09-27 19:28:...|           14.01| monthly_zone_demand|
|2026-09-27 19:28:...|2026-09-27 19:28:...|          13.716|zone_weather_dema...|
|2026-09-27 19:10:...|2026-09-27 19:10:...|           8.415|zone_weather_dema...|
+--------------------+--------------------+----------------+--------------------+



In [ ]:
spark.sql("""
    SELECT started_at,finished_at,duration_seconds,dataset
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY dataset
                ORDER BY finished_at DESC
            ) AS rn
        FROM delta.`s3a://delta-data/metadata/pipeline_executions`
    )
    WHERE rn = 1
    order by stage
""").show()

+--------------------+--------------------+----------------+--------------------+
|          started_at|         finished_at|duration_seconds|             dataset|
+--------------------+--------------------+----------------+--------------------+
|2026-09-27 19:28:...|2026-09-27 19:28:...|           6.775|air_quality_vs_de...|
|2026-09-27 19:28:...|2026-09-27 19:28:...|           26.16|average_trip_dist...|
|2026-09-27 19:27:...|2026-09-27 19:28:...|           14.01| monthly_zone_demand|
|2026-09-27 19:28:...|2026-09-27 19:28:...|          13.716|zone_weather_dema...|
|2026-09-27 19:07:...|2026-09-27 19:09:...|         100.289|integrated_taxi_t...|
|2026-09-27 19:13:...|2026-09-27 19:15:...|         123.747|         air_quality|
|2026-09-27 19:11:...|2026-09-27 19:13:...|         111.038|          taxi_trips|
|2026-09-27 19:05:...|2026-09-27 19:05:...|           9.488|          taxi_zones|
|2026-09-27 19:15:...|2026-09-27 19:15:...|           8.444|             weather|
+---------------